### Place a bounding box in order to obtain coordinates

- We should use a bounding box to cover the area that we want to extract.

In [1]:
%pip install osmnx networkx matplotlib

   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
   ----------------------------- ---------- 17.3/23.8 MB 90.1 MB/s eta 0:00:01
   ---------------------------------------- 23.8/23.8 MB 81.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ---------------------------------------- 6.3/6.3 MB 100.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 75.2 MB/s eta 0:00:00

   ---------------------------------------- 0/5 [shapely]
   -------- ------------------------------- 1/5 [pyproj]
   ---------------- ----------------------- 2/5 [pyogrio]
   ---------------- ----------------------- 2/5 [pyogrio]
   ------------------------ --------------- 3/5 [geopandas]
   ------------------------ --------------- 3/5 [geopandas]
   ---------------------------------------- 5/5 [osmnx]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import osmnx as ox

TARANGIRE_BBOX = (-4.25, 35.78, -3.70, 36.08)

G = ox.graph_from_bbox(
    bbox=TARANGIRE_BBOX,
    network_type='drive',
    retain_all=True
)
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

InsufficientResponseError: No data elements in server response. Check query location/filters and log.

In [5]:
G = ox.graph_from_place(
    "Tarangire National Park, Tanzania",
    network_type='all',
    retain_all=True
)

In [6]:
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

Nodes: 890
Edges: 2439


In [8]:
nodes, edges = ox.graph_to_gdfs(G)

print("=== EDGE ATTRIBUTES ===")
print(edges.columns.tolist())

=== EDGE ATTRIBUTES ===
['osmid', 'access', 'highway', 'maxspeed', 'oneway', 'reversed', 'length', 'geometry', 'name', 'bridge', 'service']


In [11]:
print("=== HIGHWAY TYPES ===")
print(edges['highway'].value_counts(dropna=False))

=== HIGHWAY TYPES ===
highway
track                      1452
unclassified                482
service                     237
footway                     210
residential                  30
path                         12
[service, track]              4
[service, footway]            4
[track, path]                 2
[unclassified, footway]       2
[path, residential]           2
[track, unclassified]         1
[unclassified, track]         1
Name: count, dtype: int64


In [13]:
print("=== SAMPLE EDGES ===")
print(edges.head(20))

=== SAMPLE EDGES ===
                               osmid      access       highway maxspeed  \
u          v          key                                                 
460425854  4681959723 0    474196213  permissive  unclassified       50   
           460425855  0    474196213  permissive  unclassified       50   
           4681960245 0    474189213  permissive         track      NaN   
460425855  460425854  0    474196213  permissive  unclassified       50   
           4681960245 0    474189209  permissive         track      NaN   
           460425857  0    474196213  permissive  unclassified       50   
460425857  4681960250 0    474189230  permissive         track      NaN   
           460425855  0    474196213  permissive  unclassified       50   
           2196664391 0     38811837  permissive  unclassified       50   
618532432  4660249470 0    471873526  permissive         track      NaN   
           4700139065 0    474196213  permissive  unclassified       50   
    

In [15]:
edges = edges.rename(columns={
    'highway': 'road_type',
    'length': 'length_m'
})

- We currently have OSM data for each park location.
- We don't have the type of road.
- We kinda have some suspects on what they could be, however, it seems that we need to improve our implementation or model if we want to be more precise.
- For now, in order to reduce the number of edges, we remove the ones who are road_type = 'footway' and road_type = 'path'

In [17]:
exclude_road_types = ['footway', 'path']

def is_vehicle_road(road_type) -> bool:
    return not any(r in exclude_road_types for r in road_type)

In [20]:
edges_filtered = edges[edges['road_type'].apply(is_vehicle_road)]

print(f"NODES AFTER FILTER: {len(edges_filtered)}")

NODES AFTER FILTER: 2429
